In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import gc
from collections import defaultdict
from torchmetrics.classification import BinaryAUROC

from DeepGraphDB import DeepGraphDB
from tqdm.notebook import tqdm

class KGEModel(nn.Module):
    """
    A PyTorch implementation of Knowledge Graph Embedding models.

    This class provides a unified framework for several popular KGE models,
    including TransE, DistMult, ComplEx, RotatE, and TransR. The desired model
    can be selected by passing its name as a string during initialization.

    Attributes:
        num_entities (int): The total number of unique entities in the graph.
        num_relations (int): The total number of unique relations in the graph.
        embedding_dim (int): The dimensionality of the entity and relation embeddings.
        model_name (str): The name of the KGE model to use.
        device (torch.device): The device (CPU or GPU) on which to perform computations.
    """
    def __init__(self, num_entities, num_relations, embedding_dim, model_name='TransE', margin=1.0, relation_dim=None):
        """
        Initializes the KGEModel.

        Args:
            num_entities (int): The number of entities in the knowledge graph.
            num_relations (int): The number of relations in the knowledge graph.
            embedding_dim (int): The dimension of the embeddings.
            model_name (str, optional): The name of the model to use.
                Supported models: 'TransE', 'DistMult', 'ComplEx', 'RotatE', 'TransR'.
                Defaults to 'TransE'.
            margin (float, optional): The margin for the loss function, used by TransE and RotatE.
                Defaults to 1.0.
            relation_dim (int, optional): The dimension for relation-specific projections in TransR.
                If None, defaults to embedding_dim.
        """
        super(KGEModel, self).__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.embedding_dim = embedding_dim
        self.model_name = model_name.lower()
        self.margin = margin
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        # self.device = torch.device("cpu")

        # Entity and relation embeddings
        self.entity_embeddings = nn.Embedding(num_entities, embedding_dim)
        self.relation_embeddings = nn.Embedding(num_relations, embedding_dim)

        if self.model_name == 'complex':
            # For ComplEx, relation embeddings are also complex
            self.relation_embeddings = nn.Embedding(num_relations, embedding_dim * 2)
        elif self.model_name == 'rotate':
            # For RotatE, entity embeddings are complex, relations are phases
            self.entity_embeddings = nn.Embedding(num_entities, embedding_dim * 2)
            self.relation_embeddings = nn.Embedding(num_relations, embedding_dim)
        elif self.model_name == 'transr':
            self.relation_dim = relation_dim if relation_dim is not None else embedding_dim
            self.projection_matrix = nn.Embedding(num_relations, embedding_dim * self.relation_dim)

        self.initialize_embeddings()

    def initialize_embeddings(self):
        """Initializes the embeddings with a uniform distribution."""
        nn.init.xavier_uniform_(self.entity_embeddings.weight.data)
        nn.init.xavier_uniform_(self.relation_embeddings.weight.data)
        if self.model_name == 'transr':
            nn.init.xavier_uniform_(self.projection_matrix.weight.data)

    def _transe_score(self, h, r, t):
        """Calculates the score for a triplet using the TransE model."""
        return torch.norm(h + r - t, p=2, dim=-1)
    
    # def _transr_score(self, h, r, t, proj_mat):
    #     """Calculates the score for a triplet using the TransR model."""
    #     proj_mat = proj_mat.view(-1, self.relation_dim, self.embedding_dim)
    #     h_proj = torch.bmm(h.unsqueeze(1), proj_mat).squeeze(1)
    #     t_proj = torch.bmm(t.unsqueeze(1), proj_mat).squeeze(1)
    #     return torch.norm(h_proj + r - t_proj, p=2, dim=-1)
    
    def _transr_score(self, h, r, t, r_idx): # Optimized for batch processing
        scores = h.new_empty(h.size(0), dtype=torch.float)
        uniq_r = torch.unique(r_idx)

        for r_val in uniq_r:
            mask = (r_idx == r_val)
            idx = mask.nonzero(as_tuple=False).squeeze(1)

            if idx.shape[0] == 0:
                print(r_val)
                continue

            h_i = h[idx]
            t_i = t[idx]
            r_i = r[idx]

            M_r = self.projection_matrix(r_val).view(r_i.size(1), h_i.size(1))
            # Apply projection
            h_proj = h_i @ M_r.t()
            t_proj = t_i @ M_r.t()

            # Compute scores for these triples
            scores_i = - (h_proj + r_i - t_proj).norm(p=2, dim=1)
            scores[idx] = scores_i
        
        return scores

    def _distmult_score(self, h, r, t):
        """Calculates the score for a triplet using the DistMult model."""
        return torch.sum(h * r * t, dim=-1)

    def _complex_score(self, h, r, t):
        """Calculates the score for a triplet using the ComplEx model."""
        h_real, h_imag = torch.chunk(h, 2, dim=-1)
        t_real, t_imag = torch.chunk(t, 2, dim=-1)
        r_real, r_imag = torch.chunk(r, 2, dim=-1)

        score_real = torch.sum(r_real * h_real * t_real, dim=-1)
        score_imag = torch.sum(r_real * h_imag * t_imag, dim=-1)
        score_real_imag = torch.sum(r_imag * h_real * t_imag, dim=-1)
        score_imag_real = torch.sum(r_imag * h_imag * t_real, dim=-1)

        return score_real + score_imag + score_real_imag - score_imag_real

    def _rotate_score(self, h, r, t):
        """Calculates the score for a triplet using the RotatE model."""
        h_real, h_imag = torch.chunk(h, 2, dim=-1)
        t_real, t_imag = torch.chunk(t, 2, dim=-1)
        
        # Phase of the relation
        r_phase = r / (self.embedding_dim * 0.5)
        
        r_cos = torch.cos(r_phase)
        r_sin = torch.sin(r_phase)

        re_score = (h_real * r_cos - h_imag * r_sin) - t_real
        im_score = (h_real * r_sin + h_imag * r_cos) - t_imag
        
        score = torch.stack([re_score, im_score], dim=0)
        return torch.norm(score, p=2, dim=0)

    def forward(self, triplets):
        """
        Forward pass of the KGE model.

        Args:
            triplets (torch.LongTensor): A batch of triplets of the form (head, relation, tail).

        Returns:
            torch.Tensor: The scores for the given triplets.
        """
        h_indices = triplets[:, 0]
        r_indices = triplets[:, 1]
        t_indices = triplets[:, 2]

        h = self.entity_embeddings(h_indices)
        r = self.relation_embeddings(r_indices)
        t = self.entity_embeddings(t_indices)

        if self.model_name == 'transe':
            return self._transe_score(h, r, t)
        elif self.model_name == 'distmult':
            return self._distmult_score(h, r, t)
        elif self.model_name == 'complex':
            return self._complex_score(h, r, t)
        elif self.model_name == 'rotate':
            return self._rotate_score(h, r, t)
        elif self.model_name == 'transr':
            # proj_mat = self.projection_matrix(r_indices)
            # return self._transr_score(h, r, t, proj_mat)
            return self._transr_score(h, r, t, r_indices)
        else:
            raise ValueError(f"Model {self.model_name} not supported.")

    def loss(self, positive_scores, negative_scores):
        """
        Calculates the loss for a batch of positive and negative samples.

        Args:
            positive_scores (torch.Tensor): The scores of the positive triplets.
            negative_scores (torch.Tensor): The scores of the negative triplets.

        Returns:
            torch.Tensor: The calculated loss.
        """
        if self.model_name in ['transe', 'transr', 'rotate']:
            # target = torch.tensor([-1.0], device=self.device)
            target = torch.ones_like(positive_scores)
            return F.margin_ranking_loss(positive_scores, negative_scores, target, margin=self.margin)
        elif self.model_name in ['distmult', 'complex']:
            positive_loss = F.logsigmoid(positive_scores).mean()
            negative_loss = F.logsigmoid(-negative_scores).mean()
            return - (positive_loss + negative_loss) / 2
        else:
            raise ValueError(f"Loss function for model {self.model_name} not supported.")

In [ ]:
def calculate_mrr(model, test_triplets, all_true_triplets_dict, batch_size=128):
    """
    Calculates the Mean Reciprocal Rank (MRR) using memory-efficient vectorized operations.

    Args:
        model (KGEModel): The trained KGE model.
        test_triplets (torch.LongTensor): A tensor of test triplets.
        all_true_triplets_dict (dict): A dictionary mapping (h, r) to a list of true tails.
        batch_size (int): The batch size for evaluation.

    Returns:
        float: The calculated MRR.
    """
    all_ranks = []
    model.eval()
    with torch.no_grad():
        # Special case for TransR due to its unique projection matrix per relation,
        # which is memory-intensive to batch. We process one by one.
        if model.model_name == 'transr':
            for i in tqdm(range(len(test_triplets)), desc="MRR Calculation (TransR)"):
                h, r, t = test_triplets[i]
                h_idx, r_idx, t_idx = h.item(), r.item(), t.item()
                h, r, t = h.to(model.device), r.to(model.device), t.to(model.device)
                
                h_emb = model.entity_embeddings(h)
                r_emb = model.relation_embeddings(r)

                proj_mat = model.projection_matrix(r).view(model.relation_dim, model.embedding_dim)
                all_t_emb = model.entity_embeddings.weight

                h_proj = torch.matmul(h_emb, proj_mat.transpose(0, 1))
                all_t_proj = torch.matmul(all_t_emb, proj_mat.transpose(0, 1))

                # Score is || (h_proj + r_emb) - t_proj ||
                scores = torch.cdist((h_proj + r_emb).unsqueeze(0), all_t_proj).squeeze(0)
                
                true_score = scores[t_idx]
                
                # Filter known true triplets
                temp_scores = scores.clone()
                true_tails_for_hr = all_true_triplets_dict.get((h_idx, r_idx), [])
                for true_t in true_tails_for_hr:
                    if true_t != t_idx:
                        temp_scores[true_t] = float('inf')
                
                rank = (temp_scores <= true_score).sum().item()
                all_ranks.append(1.0 / rank)
            return torch.tensor(all_ranks).mean().item()

        # General case for other models using batched, vectorized operations

        # all_t_emb = model.entity_embeddings.weight
        all_t_emb = model.ent_embeddings.weight

        if model.model_name in ['complex', 'rotate']:
            all_t_real, all_t_imag = torch.chunk(all_t_emb, 2, dim=-1)
        
        for i in tqdm(range(0, len(test_triplets), batch_size), desc="MRR Calculation"):
            batch = test_triplets[i : i + batch_size].to(model.device)
            h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
            b_size = len(batch)

            # h_emb = model.entity_embeddings(h)
            # r_emb = model.relation_embeddings(r)

            h_emb = model.ent_embeddings(h)
            r_emb = model.rel_embeddings(r)

            if model.model_name == 'transe':
                hr = h_emb + r_emb
                scores = torch.cdist(hr , all_t_emb, p=2)
            
            elif model.model_name == 'distmult':
                hr = h_emb * r_emb
                scores = torch.matmul(hr, all_t_emb.transpose(0, 1))

            elif model.model_name == 'complex':
                h_real, h_imag = torch.chunk(h_emb, 2, dim=-1)
                r_real, r_imag = torch.chunk(r_emb, 2, dim=-1)
                term1 = h_real * r_real - h_imag * r_imag
                term2 = h_real * r_imag + h_imag * r_real
                scores = torch.matmul(term1, all_t_real.transpose(0, 1)) + \
                         torch.matmul(term2, all_t_imag.transpose(0, 1))

            elif model.model_name == 'rotate':
                h_real, h_imag = torch.chunk(h_emb, 2, dim=-1)
                r_cos = torch.cos(r_emb)
                r_sin = torch.sin(r_emb)
                hr_real = h_real * r_cos - h_imag * r_sin
                hr_imag = h_real * r_sin + h_imag * r_cos
                
                # Use optimized distance calculation: ||a-b||^2 = ||a||^2 - 2a.b + ||b||^2
                hr_norm_sq = hr_real.pow(2).sum(dim=1, keepdim=True) + hr_imag.pow(2).sum(dim=1, keepdim=True)
                all_t_norm_sq = all_t_real.pow(2).sum(dim=1, keepdim=True) + all_t_imag.pow(2).sum(dim=1, keepdim=True)
                dot_product = torch.matmul(hr_real, all_t_real.transpose(0, 1)) + torch.matmul(hr_imag, all_t_imag.transpose(0, 1))
                dist_sq = hr_norm_sq - 2 * dot_product + all_t_norm_sq.transpose(0, 1)
                scores = torch.sqrt(torch.clamp(dist_sq, min=0.0))

            true_scores = scores[torch.arange(b_size), t]
            
            filter_mask = torch.zeros_like(scores, dtype=torch.bool)
            for j in range(b_size):
                true_tails = all_true_triplets_dict.get((h[j].item(), r[j].item()), [])
                if true_tails:
                    filter_mask[j, true_tails] = True
                filter_mask[j, t[j].item()] = False
            
            if model.model_name in ['distmult', 'complex']:
                scores[filter_mask] = -float('inf')
                ranks = (scores >= true_scores.unsqueeze(1)).sum(dim=1)
            else:
                scores[filter_mask] = float('inf')
                ranks = (scores <= true_scores.unsqueeze(1)).sum(dim=1)
            
            all_ranks.append(1.0 / ranks.float())
            
    return torch.cat(all_ranks).mean().item() if all_ranks else 0.0

def calculate_hits_at_k(model, test_triplets, all_true_triplets_dict, k=10, batch_size=128):
    model.eval()

    all_hits = []
    # all_t_emb = model.entity_embeddings.weight
    all_t_emb = model.ent_embeddings.weight

    with torch.no_grad():
        for i in tqdm(range(0, len(test_triplets), batch_size), desc=f"Hits@{k} Calculation"):
            batch = test_triplets[i : i + batch_size].to(model.device)
            h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
            b_size = len(batch)

            # h_emb = model.entity_embeddings(h)
            # r_emb = model.relation_embeddings(r)

            h_emb = model.ent_embeddings(h)
            r_emb = model.rel_embeddings(r)

            # if model.model_name == 'transe':
            #     scores = torch.cdist(h_emb + r_emb, all_t_emb, p=2)
            scores = torch.cdist(h_emb + r_emb, all_t_emb, p=1)

            true_scores = scores[torch.arange(b_size), t]
            
            filter_mask = torch.zeros_like(scores, dtype=torch.bool)
            for j in range(b_size):
                true_tails = all_true_triplets_dict.get((h[j].item(), r[j].item()), [])
                if true_tails:
                    filter_mask[j, true_tails] = True
                filter_mask[j, t[j].item()] = False
            
            # if model.model_name in ['distmult', 'complex']:
            #     scores[filter_mask] = -float('inf')
            #     ranks = (scores >= true_scores.unsqueeze(1)).sum(dim=1)
            # else:
            #     scores[filter_mask] = float('inf')
            #     ranks = (scores <= true_scores.unsqueeze(1)).sum(dim=1)
            scores[filter_mask] = float('inf')
            ranks = (scores <= true_scores.unsqueeze(1)).sum(dim=1)
            
            all_hits.append((ranks <= k).float())
            
    return torch.cat(all_hits).mean().item() if all_hits else 0.0

def calculate_auc(model, positive_test_triplets, negative_test_triplets):
    with torch.no_grad():
        # Get scores for positive and negative triplets
        positive_scores = model(positive_test_triplets.to(model.device))
        negative_scores = model(negative_test_triplets.to(model.device))

        # Combine predictions and binary labels
        preds = torch.cat([positive_scores, negative_scores], dim=0)
        targets = torch.cat([
            torch.ones(positive_scores.shape[0], dtype=torch.int, device=preds.device),
            torch.zeros(negative_scores.shape[0], dtype=torch.int, device=preds.device)
        ], dim=0)

        # Initialize and update AUROC metric
        auc_metric = BinaryAUROC()
        auc_metric.reset()
        auc_metric.update(preds=preds, target=targets)
        auc_value = auc_metric.compute().item()

        del positive_scores, negative_scores, preds, targets
        gc.collect()

        return auc_value

def generate_negative_samples(positive_batch, num_entities, all_true_triplets_set):
    """
    Generates a batch of negative samples, ensuring they are not true positives.
    This version is vectorized and checks for collisions, regenerating them if found.
    """
    # Start with random heads for the whole batch
    negative_heads = torch.randint(0, num_entities, (len(positive_batch),), device=positive_batch.device)
    negative_batch = positive_batch.clone()
    negative_batch[:, 0] = negative_heads

    # Find collisions: triplets that are in the set of all true triplets
    # This check is the slowest part, so we do it on CPU.
    collisions_mask = torch.tensor(
        [tuple(triplet) in all_true_triplets_set for triplet in negative_batch.cpu().numpy()]
    )
    
    # While there are still collisions, regenerate new heads for those specific triplets
    while collisions_mask.any():
        collision_indices = collisions_mask.nonzero(as_tuple=True)[0]
        num_collisions = len(collision_indices)
        
        # Generate new random heads just for the collided triplets
        new_negative_heads = torch.randint(0, num_entities, (num_collisions,), device=positive_batch.device)
        negative_batch[collision_indices, 0] = new_negative_heads
        
        # Check again for collisions, but only on the rows that were just changed
        new_check_mask = torch.tensor(
            [tuple(triplet) in all_true_triplets_set for triplet in negative_batch[collision_indices].cpu().numpy()]
        )
        
        # Update the main collision mask
        collisions_mask[collision_indices] = new_check_mask

    return negative_batch

In [ ]:
# gdb = DeepGraphDB()
# gdb.load_graph("/home/cc/PHD/dglframework/DeepKG/DeepGraphDB/graphs/primekg.bin")

# triplets = []
# ctypes = gdb.graph.canonical_etypes

# for index, ctype in enumerate(ctypes):
#     src, dst = gdb.graph.edges(etype=ctype)
#     edges_type = torch.full(src.shape, index)

#     combined_tensor = torch.stack([src, edges_type, dst], dim=1)

#     triplets.append(combined_tensor)

# triplets = torch.cat(triplets, dim=0).numpy().tolist()

# random.seed(42)  # For reproducibility
# random.shuffle(triplets)

# triplets = torch.tensor([ [ int(gdb.reverse_node_mapping[(ctypes[triple[1]][0], triple[0])]), triple[1], int(gdb.reverse_node_mapping[(ctypes[triple[1]][2], triple[2])]) ] for triple in triplets ])

# torch.save(triplets, "/home/cc/PHD/dglframework/DeepKG/triplets.pt")

triplets = torch.load("/home/cc/PHD/dglframework/DeepKG/triplets.pt")
# triplets = triplets[:50000]

In [ ]:
from dgl.data.knowledge_graph import FB15k237Dataset, FB15kDataset

data = FB15kDataset(reverse=False)[0]

src, dst = data.edges()
r = data.edata['etype']

triplets = torch.stack([src, r, dst], dim=1)

In [ ]:
num_entities = torch.concatenate((triplets[:, 0], triplets[:, 2]), dim=0).max().item()+1
num_relations = triplets[:, 1].max()+1
embedding_dim = 64
batch_size = 1000000
learning_rate = 0.01
num_epochs = 150

true_triplets_set = set(map(tuple, triplets.numpy()))
true_triplets_dict = defaultdict(list)

for h, r, t in true_triplets_set:
    true_triplets_dict[(h, r)].append(t)

# training_triplets = triplets[:int(len(triplets) * 0.7)]
# test_triplets = triplets[len(training_triplets):len(training_triplets)+int(len(triplets) * 0.20)]
# val_triplets = triplets[len(training_triplets)+len(test_triplets):]

training_triplets = triplets[data.edata['train_mask']]
test_triplets = triplets[data.edata['test_mask']]
val_triplets = triplets[data.edata['val_mask']]

# --- Test each model ---
# models_to_test = ['transe', 'transr', 'rotate', 'distmult', 'complex']
model_name = 'transe'

print(f"--- Testing {model_name} --- | num_entities: {num_entities}, num_relations: {num_relations}, embedding_dim: {embedding_dim}")
# Initialize the model
# kge_model = KGEModel(num_entities, num_relations, embedding_dim, model_name=model_name)

from DeepGraphDB.embmodels.model import TransE
from DeepGraphDB.embmodels.loss import MarginLoss

kge_model = TransE(num_entities, num_relations, embedding_dim, p_norm=1, norm_flag=True, margin=1.0, epsilon=0.01)
kge_model.model_name = model_name
kge_model.to(kge_model.device)

optimizer = torch.optim.Adam(kge_model.parameters(), lr=learning_rate)

mloss = MarginLoss(margin=1.0).to(kge_model.device)

kge_model.train()
mrr = 0.0
hits = 0.0

for epoch in tqdm(range(num_epochs)):
    # Shuffle training data each epoch
    shuffled_indices = torch.randperm(len(training_triplets))
    training_triplets_shuffled = training_triplets[shuffled_indices]
    
    total_loss = 0.0
    for i in range(0, len(training_triplets_shuffled), batch_size):
        optimizer.zero_grad()

        positive_batch = training_triplets_shuffled[i : i + batch_size].to(kge_model.device)
        positive_scores = kge_model(positive_batch)

        # negative_batch = get_negative_triplets(positive_batch.cpu(), kge_model, num_negatives=1).to(kge_model.device)
        negative_batch = generate_negative_samples(positive_batch, num_entities, true_triplets_set)
        negative_scores = kge_model(negative_batch)

        # loss = kge_model.loss(positive_scores, negative_scores)
        loss = mloss(positive_scores, negative_scores) 
        # loss = F.margin_ranking_loss(positive_scores, negative_scores, torch.ones_like(positive_scores), margin=1.0)

        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / (len(training_triplets) / batch_size)

    if (epoch+1) % 10 == 0:
        kge_model.eval()
        # TODO: mean evaluation on multiple negative sets

        del positive_batch, negative_batch, positive_scores, negative_scores
        torch.cuda.empty_cache()
        gc.collect()
        # negative_test_triplets = get_negative_triplets(test_triplets, kge_model, num_negatives=1)
        shuffled_indices = torch.randint(0, len(val_triplets), (len(val_triplets)//10,))
        negative_test_triplets = generate_negative_samples(val_triplets, num_entities, true_triplets_set)

        #if (epoch+1) % 40 == 0:
            #mrr = calculate_mrr(kge_model, test_triplets[shuffled_indices], true_triplets_dict, batch_size=10000)
        mrr = calculate_mrr(kge_model, val_triplets[shuffled_indices], true_triplets_dict, batch_size=10000) #TODO: optimize the calulcation for transr (also transe if is possibile) in batch!
        auc = calculate_auc(kge_model, val_triplets, negative_test_triplets)
        hits = calculate_hits_at_k(kge_model, val_triplets[shuffled_indices], true_triplets_dict, k=10, batch_size=10000)

        del negative_test_triplets, shuffled_indices
        torch.cuda.empty_cache()
        gc.collect()

        kge_model.train()

        # print(f"Evaluation MRR: {mrr:.4f}")
        print(f"Epoch {epoch+1} finished. Average Loss: {avg_loss:.4f} - Evaluation AUC: {auc:.4f} - Evaluation HITS@10: {hits:.4f} - Evaluation MRR: {mrr:.4f}\n")

In [ ]:
proj = nn.Embedding(50, 128 * 128, device="cuda")
h_e = torch.randn(5000000, 128, device="cuda")
r_e = torch.randn(5000000, 128, device="cuda")
t_e = torch.randn(5000000, 128, device="cuda")

r = torch.randint(0, 50, (5000000,), device="cuda")

scores = h_e.new_empty(h_e.size(0), dtype=torch.float)
uniq_r = torch.unique(r)

for r_val in uniq_r:
    mask = (r == r_val)
    idx = mask.nonzero(as_tuple=False).squeeze(1)

    h_i = h_e[idx]
    t_i = t_e[idx]
    r_i = r_e[idx]

    # Projection matrix for this relation
    M_r = proj(r_val).view(r_i.size(1), h_i.size(1))
    # Apply projection
    h_proj = h_i @ M_r.t()
    t_proj = t_i @ M_r.t()

    # Compute scores for these triples
    scores_i = - (h_proj + r_i - t_proj).norm(p=2, dim=1)
    scores[idx] = scores_i

scores.shape

In [ ]:
from pykeen.pipeline import pipeline

result = pipeline(
    dataset='FB15k',
    model='TransE',
    training_loop='sLCWA',
    negative_sampler='basic',
    evaluator='RankBasedEvaluator'
)

In [ ]:
result.save_to_directory('fbk_transe')

In [ ]:
model = result.model

entity_representation_modules = model.entity_representations
relation_representation_modules = model.relation_representations